# Chess.com Data Collection

This notebook collects public chess data from the Chess.com PubAPI.

The goal of this project is to build a personal chess analytics pipeline for:
- performance analysis
- rating progression
- opening analysis
- behavioral analytics
- visualization projects

Data source:
https://api.chess.com/pub/

## Imports

In [1]:
import requests
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from datetime import datetime
import time

## User configuration

The Chess.com username is defined once in `src/config.py` and imported here.

In [2]:
PROJECT_ROOT = Path().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import USERNAME

## Project paths

In [3]:
RAW_DATA_DIR = PROJECT_ROOT / "dataset" / "raw" / USERNAME

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_DIR

WindowsPath('D:/Data Analisi/notebooks/scacchi/chesscom-personal-analysis/dataset/raw/elmurie')

# API helper

Chess.com requires a valid User-Agent in order to avoid 403/rate limiting.

In [4]:
headers = {
    "User-Agent": "chess-analytics-project"
}

BASE_URL = f"https://api.chess.com/pub/player/{USERNAME}"


def get_json(url):

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Request failed: {url}")
        print(response.status_code)
        return None

    return response.json()

## Download player profile

In [5]:
profile = get_json(BASE_URL)

profile_df = pd.json_normalize(profile)

profile_df.T.head(20)

Request failed: https://api.chess.com/pub/player/elmurie
429


NotImplementedError: 

In [13]:
profile_df.to_csv(
    RAW_DATA_DIR / "profile.csv",
    index=False
)

## Download player stats

In [14]:
stats = get_json(f"{BASE_URL}/stats")

stats_df = pd.json_normalize(stats)

stats_df.T.head(30)

,0
chess_daily.last.rating,712
chess_daily.last.date,1773380333
chess_daily.last.rd,129
chess_daily.best.rating,1041
chess_daily.best.date,1607205002
chess_daily.best.game,https://www.chess.com/game/daily/624828741
chess_daily.record.win,34
chess_daily.record.loss,39
chess_daily.record.draw,0
chess_daily.record.time_per_move,28055


In [15]:
stats_df.to_csv(
    RAW_DATA_DIR / "stats.csv",
    index=False
)

## Game Archives

Chess.com stores games in monthly archives.

The API first returns a list of archive URLs,
then each archive contains the games played during that month.

In [16]:
archives_data = get_json(
    f"{BASE_URL}/games/archives"
)

archives = archives_data["archives"]

len(archives)

67

## Download all games

In [17]:
all_games = []

for archive in archives:

    data = get_json(archive)

    if not data:
        continue

    for game in data.get("games", []):

        all_games.append({

            "date": (
                datetime.fromtimestamp(
                    game.get("end_time")
                ).strftime("%Y-%m-%d %H:%M:%S")
                if game.get("end_time")
                else None
            ),

            "url": game.get("url"),

            "time_class": game.get("time_class"),
            "time_control": game.get("time_control"),

            "white": game.get("white", {}).get("username"),
            "black": game.get("black", {}).get("username"),

            "white_rating": game.get("white", {}).get("rating"),
            "black_rating": game.get("black", {}).get("rating"),

            "white_result": game.get("white", {}).get("result"),
            "black_result": game.get("black", {}).get("result"),

            "eco": game.get("eco"),

            "pgn": game.get("pgn")
        })

Request failed: https://api.chess.com/pub/player/elmurie/games/2021/01
429
Request failed: https://api.chess.com/pub/player/elmurie/games/2021/03
429
Request failed: https://api.chess.com/pub/player/elmurie/games/2021/04
429
Request failed: https://api.chess.com/pub/player/elmurie/games/2021/05
429
Request failed: https://api.chess.com/pub/player/elmurie/games/2022/05
429
Request failed: https://api.chess.com/pub/player/elmurie/games/2022/06
429
Request failed: https://api.chess.com/pub/player/elmurie/games/2023/04
429
Request failed: https://api.chess.com/pub/player/elmurie/games/2023/05
429
Request failed: https://api.chess.com/pub/player/elmurie/games/2023/06
429
Request failed: https://api.chess.com/pub/player/elmurie/games/2023/07
429
Request failed: https://api.chess.com/pub/player/elmurie/games/2024/06
429
Request failed: https://api.chess.com/pub/player/elmurie/games/2024/07
429
Request failed: https://api.chess.com/pub/player/elmurie/games/2024/08
429
Request failed: https://a

## Create Dataframe

In [18]:
games_df = pd.DataFrame(all_games)

games_df.head()

,date,url,time_class,time_control,white,black,white_rating,black_rating,white_result,black_result,eco,pgn
0,2020-11-10 15:30:24,https://www.chess.com/game/live/5719328865,rapid,600,elmurie,TheMrNoName,213,412,resigned,win,https://www.chess.com/openings/Polish-Opening-...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."
1,2020-11-10 15:46:19,https://www.chess.com/game/live/5719418873,rapid,600,smackersmashbot,elmurie,216,345,resigned,win,https://www.chess.com/openings/Kings-Pawn-Open...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."
2,2020-11-10 16:09:31,https://www.chess.com/game/live/5719482469,rapid,600,amkh98,elmurie,371,256,win,checkmated,https://www.chess.com/openings/Kings-Pawn-Open...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."
3,2020-11-10 17:39:54,https://www.chess.com/game/live/5720005454,rapid,600,elmurie,callumfindlay4,193,330,checkmated,win,https://www.chess.com/openings/Vienna-Game-Max...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."
4,2020-11-10 17:51:42,https://www.chess.com/game/live/5720050101,rapid,600,callumfindlay4,elmurie,291,277,resigned,win,https://www.chess.com/openings/Caro-Kann-Defen...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."


## Dataset overview

The dataset looks pretty good!

In [19]:
games_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16139 entries, 0 to 16138
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   date          16139 non-null  object
 1   url           16139 non-null  object
 2   time_class    16139 non-null  object
 3   time_control  16139 non-null  object
 4   white         16139 non-null  object
 5   black         16139 non-null  object
 6   white_rating  16139 non-null  int64 
 7   black_rating  16139 non-null  int64 
 8   white_result  16139 non-null  object
 9   black_result  16139 non-null  object
 10  eco           16139 non-null  object
 11  pgn           16139 non-null  object
dtypes: int64(2), object(10)
memory usage: 1.5+ MB


In [20]:
games_df.describe(include="all")

,date,url,time_class,time_control,white,black,white_rating,black_rating,white_result,black_result,eco,pgn
count,16139,16139,16139,16139,16139,16139,16139.000000,16139.000000,16139,16139,16139,16139
unique,16139,16139,4,11,7970,7958,NaN,NaN,10,10,1286,16139
top,2020-11-10 15:30:24,https://www.chess.com/game/live/5719328865,blitz,180,elmurie,elmurie,NaN,NaN,win,win,https://www.chess.com/openings/Vienna-Game,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."
freq,1,1,15238,12437,8061,8078,NaN,NaN,8193,7427,743,1
mean,NaN,NaN,NaN,NaN,NaN,NaN,629.735424,628.634860,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,123.612664,122.691928,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,100.000000,100.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,563.000000,562.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,649.000000,649.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,716.000000,715.000000,NaN,NaN,NaN,NaN


In [21]:
games_df["time_class"].value_counts()

time_class
blitz     15238
bullet      449
rapid       388
daily        64
Name: count, dtype: int64

## Opponents list

We can extract the opponents' country, but we need to extract the profiles first

### Opponent normalisation 

In [22]:
games_df["opponent"] = np.where(
    games_df["white"] == USERNAME,
    games_df["black"],
    games_df["white"]
)

games_df["opponent_clean"] = (
    games_df["opponent"]
    .dropna()
    .str.lower()
    .str.strip()
)

### Unique opponents

In [23]:
opponents = (
    games_df["opponent_clean"]
    .dropna()
    .unique()
)

print(
    f"Unique opponents: {len(opponents)}"
)

Unique opponents: 15634


### Save unique opponents

In [24]:
opponents_df = pd.DataFrame({
    "opponent": opponents
})

opponents_path = (
    RAW_DATA_DIR
    / "unique_opponents.csv"
)

opponents_df.to_csv(
    opponents_path,
    index=False
)

print(
    f"Saved: {opponents_path}"
)

Saved: D:\Data Analisi\notebooks\scacchi\chesscom-personal-analysis\dataset\raw\elmurie\unique_opponents.csv


### Opponent enrichment cache

In [25]:
profiles_path = (
    RAW_DATA_DIR
    / "opponent_profiles.csv"
)

### Load existing cache if present

In [26]:
if profiles_path.exists():

    existing_profiles = pd.read_csv(
        profiles_path
    )

    print(
        f"Loaded existing profiles: {len(existing_profiles)}"
    )

else:

    existing_profiles = pd.DataFrame()

    print(
        "No existing profile cache found."
    )

Loaded existing profiles: 8652


### Already downloaded users

In [27]:
if not existing_profiles.empty:

    downloaded_users = set(
        existing_profiles["opponent_clean"]
    )

else:

    downloaded_users = set()

### Safer get_json()

In [28]:
def get_json(url):

    try:

        response = requests.get(
            url,
            headers=headers,
            timeout=10
        )

        if response.status_code != 200:

            return None

        return response.json()

    except Exception as e:

        print(f"ERROR: {url}")

        print(e)

        return None

### Opponent profile enrichment

In [29]:
new_profiles = []

total = len(opponents_df)

for i, opponent in enumerate(
    opponents_df["opponent"]
):

    # skip existing users
    if opponent in downloaded_users:

        continue

    try:

        OPPONENT_URL = (
            f"https://api.chess.com/pub/player/{opponent}"
        )

        profile = get_json(OPPONENT_URL)

        if not profile:

            continue

        country_url = profile.get("country")

        country_code = None

        if country_url:

            country_code = (
                country_url
                .split("/")[-1]
            )

        new_profiles.append({

            "opponent_clean": opponent,

            "country_code": country_code,

            "title": profile.get("title"),

            "followers": profile.get("followers"),

            "joined": profile.get("joined"),

            "last_online": profile.get("last_online")
        })

        # progress
        if i % 100 == 0:

            print(
                f"{i}/{total}"
            )

        # incremental save
        if i % 100 == 0:

            temp_df = pd.concat([

                existing_profiles,

                pd.DataFrame(new_profiles)

            ]).drop_duplicates(
                subset="opponent_clean"
            )

            temp_df.to_csv(
                profiles_path,
                index=False
            )

            print(
                f"Incremental save at {i}"
            )

        time.sleep(0.5)

    except Exception as e:

        print(
            f"ERROR on {opponent}"
        )

        print(e)

        continue

5500/15634
Incremental save at 5500
6100/15634
Incremental save at 6100


KeyboardInterrupt: 

### Final save

In [ ]:
final_profiles = pd.concat([

    existing_profiles,

    pd.DataFrame(new_profiles)

]).drop_duplicates(
    subset="opponent_clean"
)

final_profiles.to_csv(
    profiles_path,
    index=False
)

print(
    f"Final profiles saved: {len(final_profiles)}"
)

## Save dataset

In [ ]:
games_df.to_csv(
    RAW_DATA_DIR / "games.csv",
    index=False
)

print("Dataset saved.")

# Next Steps

The next notebook will focus on:
- cleaning timestamps
- extracting player-side information
- parsing openings
- handling missing values
- feature engineering